# Spoken English Grammar Scoring – Final Solution

**Candidate:** Kiran G  
**Task:** Predict a continuous grammar proficiency score from spoken English audio.  
**Score range:** 0–5

This notebook documents the final end-to-end methodology, preprocessing, feature extraction, modeling, cross-validation, evaluation, interpretability, and inference pipeline used for the challenge.


## 1. Problem Statement

The objective is to predict a continuous grammar proficiency score for each spoken-English audio recording.

The solution uses a multimodal approach combining:
- self-supervised speech representations,
- grammar-aware transcript representations,
- handcrafted acoustic features,
- and an audio–text cross-attention model.


## 2. Dataset Overview

The dataset contains:
- a labeled training set of WAV recordings,
- an unlabeled test set,
- `train.csv`,
- `test.csv`.

The target score lies in the range **0 to 5**.


## 3. Evaluation Metric

Model performance is evaluated using:
- **RMSE (Root Mean Squared Error)**
- **Pearson correlation**

Lower RMSE is better, while higher Pearson correlation is better.

The challenge additionally requires the training RMSE to be reported in the final notebook.


## 4. Data Preprocessing

Main preprocessing steps:
1. Audio loaded at 16 kHz mono.
2. Long recordings split into chunks where required by pretrained speech encoders.
3. Transcript generation performed using Whisper.
4. Features aligned using filename rather than row position.
5. Predictions clipped to the valid score range `[0, 5]`.
6. Cross-validation performed using stratified score groups.


## 5. Speech Transcription

Whisper `small.en` was used to generate English transcripts from the training and test audio.

Transcript-based features were used for grammar-aware modeling and audio–text cross-modal learning.


## 6. Feature Extraction


### 6.1 WavLM Speech Embeddings

`microsoft/wavlm-base-plus` was used as a frozen pretrained speech encoder. Multiple transformer layers were evaluated. Layer 9 produced the strongest validation performance. For the main WavLM expert, audio was split into chunks, hidden representations were extracted, temporal mean and standard deviation were pooled, and an RBF SVR was trained on the resulting representation.


### 6.2 HuBERT Speech Embeddings

`facebook/hubert-base-ls960` was used as a second pretrained speech representation. HuBERT provides a complementary self-supervised acoustic representation to WavLM.


### 6.3 Grammar-Aware Text Embeddings

A CoLA-trained RoBERTa model (`textattack/roberta-base-CoLA`) was used to generate grammar-sensitive transcript embeddings. The transcript representation used contextual hidden states and pooling to capture grammatical acceptability information.


### 6.4 Handcrafted Acoustic Features

The handcrafted acoustic branch includes features such as RMS energy, zero crossing rate, spectral centroid, spectral bandwidth, rolloff, silence ratio, and MFCC statistics. An ensemble of RBF SVR and Extra Trees was used for this branch.


### 6.5 Cross-Modal Audio–Text Features

A small cross-attention neural network was trained over frozen WavLM temporal audio representations and grammar-aware sentence embeddings. The model used bidirectional audio↔text cross-attention, regression loss, auxiliary score classification, and ranking loss. This model was later blended with the multimodal expert ensemble.


## 7. Model Architecture


### 7.1 Individual Models

| Component | Main model |
|---|---|
| WavLM Layer 9 | RBF SVR |
| HuBERT | RBF SVR |
| Grammar embedding | RBF SVR |
| Acoustic features | SVR + Extra Trees |
| Cross-modal model | Neural cross-attention |


### 7.2 Multimodal Ensemble

The strong multimodal ensemble combined:
- **50% WavLM Layer 9**
- **20% HuBERT**
- **20% grammar embedding**
- **10% handcrafted audio**

This model achieved a public leaderboard score of **0.3818**.


### 7.3 Cross-Attention Model

The cross-attention model combined temporal audio features with grammar-aware text sequence representations. A later ensemble combined:
- **70% multimodal ensemble**
- **30% cross-attention model**

This became the strongest public leaderboard submission with a score of **0.3603**.


## 8. Cross-Validation Strategy

A **5-fold StratifiedKFold** strategy was used. Continuous scores were grouped into broad score regions for stratification: ≤2, >2 and ≤3, >3 and ≤4, and >4. All trainable preprocessing steps such as scaling were fit inside each training fold to reduce leakage.


## 9. Training Results

Selected out-of-fold results from the development process:

| Model | OOF RMSE | Pearson |
|---|---:|---:|
| WavLM Layer 9 | ~0.566 | ~0.890 |
| HuBERT | ~0.602 | ~0.874 |
| Grammar embedding | ~0.815 | ~0.753 |
| V12 cross-modal | ~0.552 | ~0.896 |
| V11 multimodal ensemble | ~0.540 | ~0.910 |
| V13 blend | ~0.518 | ~0.914 |


## 10. Training RMSE

The final selected V13-style ensemble achieved an out-of-fold training RMSE of approximately **0.518**. This value is reported from cross-validated out-of-fold predictions rather than fitting and evaluating on the same training samples.


## 11. Interpretability and Analysis

Several analyses were used to understand model behavior:
- comparison of RMSE and Pearson correlation across feature families,
- error-correlation analysis between experts,
- inspection of ensemble weights,
- comparison of acoustic and grammar branches,
- analysis of whether new models contributed complementary errors.

One important finding was that the cross-attention model was weaker than the multimodal ensemble alone, but its errors were sufficiently different to improve the final blended model.


## 12. Final Inference Pipeline

1. Load each test audio file.
2. Extract WavLM features.
3. Extract HuBERT features.
4. Generate transcript with Whisper.
5. Extract grammar-aware transcript embeddings.
6. Extract handcrafted acoustic features.
7. Run the cross-modal audio–text model.
8. Generate multimodal prediction.
9. Blend multimodal and cross-attention predictions.
10. Clip prediction to `[0, 5]`.
11. Write predictions in the order of `test.csv`.


## 13. Final Submission

The strongest public leaderboard submission was **V13 — 70% V11 multimodal + 30% V12 cross-attention**, with a public leaderboard score of **0.3603**. A second selected submission was the V11 multimodal ensemble with a score of **0.3818**.


## 14. Limitations and Future Improvements

Limitations include the small labeled dataset, mostly frozen pretrained encoders, ASR dependence, and the fact that local CV and the public leaderboard did not always move together. Future work could explore larger labeled data, carefully regularized encoder fine-tuning, stronger ordinal objectives, calibrated multi-task learning, repeated or nested cross-validation, and pronunciation-specific pretrained representations.


## 15. Conclusion

The strongest approach was a multimodal system combining acoustic, speech-representation, grammar-aware text, and cross-modal features. The best public submission achieved **0.3603**, improving substantially over the early baselines. The final solution prioritizes complementary model errors rather than relying on a single representation.


In [ ]:
# Optional: basic environment check
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
